# 02 — ELO team strength

`elo_diff` (home ELO − away ELO before the match) controls for team strength in the models.
We use ELO from results instead of betting odds: odds from the no-fans period already price in the missing home advantage, so they would "eat" part of the effect we want to measure.

The ELO logic lives in `src/elo.py`. Seasons 2010/11–2016/17 only warm the ratings up.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../src")     # so we can import our own module
from elo import compute_elo

PROCESSED_DIR = Path("../data/processed")
pd.set_option("display.width", 200)

matches = pd.read_csv(PROCESSED_DIR / "matches.csv", parse_dates=["Date"])
print(matches.shape)

(4940, 25)


## 1. Choosing K and home advantage

ELO has two settings:
- `k`: how much one result moves the rating (bigger = reacts faster, but noisier),
- `home_advantage`: rating points added to the home team when computing the expected result.

We pick them by how well ELO predicts results, using **only pre-pandemic seasons 2011/12–2018/19** (2010/11 is skipped because everyone starts at 1500).
That way the choice cannot be influenced by the no-fans period we study.

Measure of fit: mean squared error between the actual score (1 / 0.5 / 0) and the expected score. Lower is better.

In [2]:
results = []

for k in [10, 20, 30, 40]:
    for home_advantage in [0, 30, 60, 90]:
        rated, _ = compute_elo(matches, k=k, home_advantage=home_advantage)
        tuning = rated[rated["season_start"].between(2011, 2018)]

        actual = (tuning["FTHG"] > tuning["FTAG"]) * 1.0 + (tuning["FTHG"] == tuning["FTAG"]) * 0.5
        error = ((actual - tuning["expected_home"]) ** 2).mean()

        results.append({"k": k, "home_advantage": home_advantage, "mse": round(error, 5)})

fit = pd.DataFrame(results)
fit.pivot(index="k", columns="home_advantage", values="mse")

home_advantage,0,30,60,90
k,,,,
10,0.15904,0.15442,0.15297,0.15450
20,0.15852,0.15388,0.15225,0.15341
30,0.15948,0.15479,0.15303,0.15402
40,0.16092,0.15616,0.15430,0.15515


In [3]:
best = fit.sort_values("mse").iloc[0]
K = int(best["k"])
HOME_ADVANTAGE = int(best["home_advantage"])
print(f"Best: k = {K}, home_advantage = {HOME_ADVANTAGE}")

Best: k = 20, home_advantage = 60


## 2. Compute ELO

In [4]:
matches, season_end = compute_elo(matches, k=K, home_advantage=HOME_ADVANTAGE)
matches[["season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "home_elo", "away_elo", "elo_diff"]].tail()

,season,Date,HomeTeam,AwayTeam,FTHG,FTAG,home_elo,away_elo,elo_diff
4935,2022/23,2023-05-28,Everton,Bournemouth,1,0,1416.399185,1380.033825,36.365360
4936,2022/23,2023-05-28,Leeds,Tottenham,1,4,1385.396800,1539.223584,-153.826785
4937,2022/23,2023-05-28,Leicester,West Ham,2,1,1416.018433,1463.384430,-47.365996
4938,2022/23,2023-05-28,Man United,Fulham,2,1,1606.728360,1432.576146,174.152214
4939,2022/23,2023-05-28,Southampton,Liverpool,4,4,1335.569183,1672.689731,-337.120548


## 3. Checkpoint: does the ranking look sensible?

From PLAN.md: the ranking at the end of 2018/19 should look sensible (champion near the top).
We compare it with the actual league table, rebuilt from the same results (3 points for a win, 1 for a draw).

In [5]:
def league_table(season_matches):
    """Points per team from match results."""
    home_points = season_matches["FTHG"].gt(season_matches["FTAG"]) * 3 + season_matches["FTHG"].eq(season_matches["FTAG"]) * 1
    away_points = season_matches["FTAG"].gt(season_matches["FTHG"]) * 3 + season_matches["FTHG"].eq(season_matches["FTAG"]) * 1
    points = pd.concat([
        pd.DataFrame({"team": season_matches["HomeTeam"], "points": home_points}),
        pd.DataFrame({"team": season_matches["AwayTeam"], "points": away_points}),
    ])
    return points.groupby("team")["points"].sum()


SEASON = 2018
table = season_end[season_end["season_start"] == SEASON].set_index("team")[["elo"]].round(0)
table["points"] = league_table(matches[matches["season_start"] == SEASON])
table["elo_rank"] = table["elo"].rank(ascending=False).astype(int)
table["league_rank"] = table["points"].rank(ascending=False, method="min").astype(int)
table

,elo,points,elo_rank,league_rank
team,,,,
Man City,1793.0,98,1,1
Liverpool,1745.0,97,2,2
Chelsea,1623.0,72,3,3
Tottenham,1609.0,71,4,4
Man United,1606.0,66,5,6
Arsenal,1587.0,70,6,5
Everton,1513.0,54,7,8
Wolves,1486.0,57,8,7
Crystal Palace,1486.0,49,8,12


In [6]:
# Spearman correlation: 1 = ELO order identical to the league table
for season in range(2017, 2023):
    t = season_end[season_end["season_start"] == season].set_index("team")
    t["points"] = league_table(matches[matches["season_start"] == season])
    print(season, "/", season + 1, " rank correlation ELO vs points:", round(t["elo"].corr(t["points"], method="spearman"), 2))

2017 / 2018  rank correlation ELO vs points: 0.92
2018 / 2019  rank correlation ELO vs points: 0.98
2019 / 2020  rank correlation ELO vs points: 0.97
2020 / 2021  rank correlation ELO vs points: 0.97
2021 / 2022  rank correlation ELO vs points: 0.97
2022 / 2023  rank correlation ELO vs points: 0.95


## 4. Does `elo_diff` predict results?

If ELO measures strength, the home team should win more often as `elo_diff` grows. Checked on the analysis sample.

In [7]:
analysis = matches[matches["season_start"] >= 2017].copy()
analysis["elo_diff_group"] = pd.qcut(analysis["elo_diff"], 5)
analysis["home_win"] = (analysis["FTHG"] > analysis["FTAG"]).astype(int)

analysis.groupby("elo_diff_group", observed=True).agg(
    matches=("home_win", "size"),
    home_win_rate=("home_win", "mean"),
    mean_card_diff=("card_diff", "mean"),
).round(3)

,matches,home_win_rate,mean_card_diff
elo_diff_group,,,
"(-490.81199999999995, -142.278]",456,0.202,-0.254
"(-142.278, -39.364]",456,0.333,-0.055
"(-39.364, 39.001]",456,0.425,0.083
"(39.001, 143.157]",456,0.533,0.397
"(143.157, 463.443]",456,0.737,0.482


## 5. Save

In [8]:
matches = matches.sort_values(["season_start", "Date"], kind="stable")
matches = matches.drop(columns="expected_home")
matches.to_csv(PROCESSED_DIR / "matches_elo.csv", index=False)
print("Saved", matches.shape, "to data/processed/matches_elo.csv")
print("Missing elo_diff:", matches["elo_diff"].isna().sum())

Saved (4940, 28) to data/processed/matches_elo.csv
Missing elo_diff: 0
